# SC-14-Cross-Chain - Interoperabilite Cross-Chain

**Navigation** : [Index](../README.md) | [<< Solana](SC-13-Solana-Anchor.ipynb) | [Capstone >>](../05-Capstone/SC-15-Final-Project.ipynb)

---

## Objectifs d'apprentissage

1. Comprendre les enjeux de l'**interoperabilite**
2. Utiliser **Chainlink CCIP** pour les cross-chain messages
3. Implementer un **bridge** simple
4. Comprendre les **securite** cross-chain

### Duree estimee : 45 minutes

---

## 1. Enjeux Cross-Chain

L'interoperabilite entre blockchains est un defi majeur.

In [1]:
# Concepts cross-chain
print("""
DEFIS CROSS-CHAIN

| Probleme          | Description                         |
|------------------|-------------------------------------|
| Finalite         | Verifier la finalite sur la source  |
| Securite         | Empecher les double-spend           |
| Latence          | Temps de confirmation               |
| Cout             | Gas sur les deux chains             |
| Oracle           | Obtenir l'etat d'une autre chain    |

SOLUTIONS:
- Lock & Mint     : Lock tokens A, mint wrapped A' sur B
- Burn & Mint     : Burn wrapped A', unlock A sur source
- Liquidity Pools : Swap via pools sur chaque chain
- Light Clients   : Verifier les headers distamment

PROTOCOLS POPULAIRES:
- Chainlink CCIP  : Oracle-based messaging
- LayerZero       : Omnichain interoperability
- Axelar          : Cross-chain gateway
- Wormhole        : Guardian network
- Hyperlane       : Permissionless interoperability
""")


DEFIS CROSS-CHAIN

| Probleme          | Description                         |
|------------------|-------------------------------------|
| Finalite         | Verifier la finalite sur la source  |
| Securite         | Empecher les double-spend           |
| Latence          | Temps de confirmation               |
| Cout             | Gas sur les deux chains             |
| Oracle           | Obtenir l'etat d'une autre chain    |

SOLUTIONS:
- Lock & Mint     : Lock tokens A, mint wrapped A' sur B
- Burn & Mint     : Burn wrapped A', unlock A sur source
- Liquidity Pools : Swap via pools sur chaque chain
- Light Clients   : Verifier les headers distamment

PROTOCOLS POPULAIRES:
- Chainlink CCIP  : Oracle-based messaging
- LayerZero       : Omnichain interoperability
- Axelar          : Cross-chain gateway
- Wormhole        : Guardian network
- Hyperlane       : Permissionless interoperability



---

## 2. Chainlink CCIP

In [2]:
# Chainlink CCIP Receiver
CCIP_RECEIVER = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

import "@chainlink/contracts/src/v0.8/ccip/interfaces/IAny2EVMMessageReceiver.sol";
import "@chainlink/contracts/src/v0.8/ccip/libraries/Client.sol";
import "@chainlink/contracts/src/v0.8/ccip/applications/CCIPReceiver.sol";

contract CrossChainReceiver is CCIPReceiver {
    // Message recu
    struct Message {
        uint64 sourceChainSelector;
        address sender;
        bytes data;
    }

    Message[] public messages;

    event MessageReceived(
        uint64 indexed sourceChainSelector,
        address sender,
        bytes data
    );

    constructor(address router) CCIPReceiver(router) {}

    // Callback CCIP
    function _ccipReceive(
        Client.Any2EVMMessage memory message
    ) internal override {
        uint64 sourceChainSelector = message.sourceChainSelector;
        address sender = abi.decode(message.sender, (address));
        bytes memory data = message.data;

        messages.push(Message({
            sourceChainSelector: sourceChainSelector,
            sender: sender,
            data: data
        }));

        emit MessageReceived(sourceChainSelector, sender, data);
    }

    function getMessages() external view returns (Message[] memory) {
        return messages;
    }
}
'''

print("Receiver CCIP:")
print(CCIP_RECEIVER)

Receiver CCIP:

// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

import "@chainlink/contracts/src/v0.8/ccip/interfaces/IAny2EVMMessageReceiver.sol";
import "@chainlink/contracts/src/v0.8/ccip/libraries/Client.sol";
import "@chainlink/contracts/src/v0.8/ccip/applications/CCIPReceiver.sol";

contract CrossChainReceiver is CCIPReceiver {
    // Message recu
    struct Message {
        uint64 sourceChainSelector;
        address sender;
        bytes data;
    }

    Message[] public messages;

    event MessageReceived(
        uint64 indexed sourceChainSelector,
        address sender,
        bytes data
    );

    constructor(address router) CCIPReceiver(router) {}

    // Callback CCIP
    function _ccipReceive(
        Client.Any2EVMMessage memory message
    ) internal override {
        uint64 sourceChainSelector = message.sourceChainSelector;
        address sender = abi.decode(message.sender, (address));
        bytes memory data = message.data;

        messages.push(M

In [3]:
# Chainlink CCIP Sender
CCIP_SENDER = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

import "@chainlink/contracts/src/v0.8/ccip/interfaces/IRouterClient.sol";
import "@chainlink/contracts/src/v0.8/ccip/libraries/Client.sol";
import "@chainlink/contracts/src/v0.8/shared/access/ConfirmedOwner.sol";

contract CrossChainSender is ConfirmedOwner {
    IRouterClient private immutable router;

    event MessageSent(bytes32 messageId);

    constructor(address _router) ConfirmedOwner(msg.sender) {
        router = IRouterClient(_router);
    }

    // Envoyer un message cross-chain
    function sendMessage(
        uint64 destinationChainSelector,
        address receiver,
        bytes calldata data,
        address feeToken
    ) external payable returns (bytes32) {
        // Construire le message
        Client.EVM2AnyMessage memory message = Client.EVM2AnyMessage({
            receiver: abi.encode(receiver),
            data: data,
            tokenAmounts: new Client.EVMTokenAmount[](0),
            extraArgs: Client._defaultExtraArgs(),
            feeToken: feeToken
        });

        // Calculer les frais
        uint256 fees = router.getFee(destinationChainSelector, message);
        require(msg.value >= fees, "Insufficient fee");

        // Envoyer via CCIP
        bytes32 messageId = router.ccipSend{value: fees}(
            destinationChainSelector,
            message
        );

        emit MessageSent(messageId);
        return messageId;
    }

    // Envoyer tokens + message
    function sendMessageWithToken(
        uint64 destinationChainSelector,
        address receiver,
        bytes calldata data,
        address token,
        uint256 amount,
        address feeToken
    ) external payable returns (bytes32) {
        // Transfer tokens to this contract
        IERC20(token).transferFrom(msg.sender, address(this), amount);
        IERC20(token).approve(address(router), amount);

        Client.EVMTokenAmount[] memory tokenAmounts = 
            new Client.EVMTokenAmount[](1);
        tokenAmounts[0] = Client.EVMTokenAmount({
            token: token,
            amount: amount
        });

        Client.EVM2AnyMessage memory message = Client.EVM2AnyMessage({
            receiver: abi.encode(receiver),
            data: data,
            tokenAmounts: tokenAmounts,
            extraArgs: Client._defaultExtraArgs(),
            feeToken: feeToken
        });

        uint256 fees = router.getFee(destinationChainSelector, message);
        require(msg.value >= fees, "Insufficient fee");

        bytes32 messageId = router.ccipSend{value: fees}(
            destinationChainSelector,
            message
        );

        emit MessageSent(messageId);
        return messageId;
    }
}

interface IERC20 {
    function transferFrom(address, address, uint256) external returns (bool);
    function approve(address, uint256) external returns (bool);
}
'''

print("Sender CCIP:")
print(CCIP_SENDER)

Sender CCIP:

// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

import "@chainlink/contracts/src/v0.8/ccip/interfaces/IRouterClient.sol";
import "@chainlink/contracts/src/v0.8/ccip/libraries/Client.sol";
import "@chainlink/contracts/src/v0.8/shared/access/ConfirmedOwner.sol";

contract CrossChainSender is ConfirmedOwner {
    IRouterClient private immutable router;

    event MessageSent(bytes32 messageId);

    constructor(address _router) ConfirmedOwner(msg.sender) {
        router = IRouterClient(_router);
    }

    // Envoyer un message cross-chain
    function sendMessage(
        uint64 destinationChainSelector,
        address receiver,
        bytes calldata data,
        address feeToken
    ) external payable returns (bytes32) {
        // Construire le message
        Client.EVM2AnyMessage memory message = Client.EVM2AnyMessage({
            receiver: abi.encode(receiver),
            data: data,
            tokenAmounts: new Client.EVMTokenAmount[](0),
            

---

## 3. Chain Selectors

In [4]:
# Chain selectors CCIP
print("""
CHAIN SELECTORS CHAINLINK CCIP

| Chain              | Selector              |
|--------------------|-----------------------|
| Ethereum Mainnet   | 5009297550715157      |
| Ethereum Sepolia   | 16015286601757825753  |
| Arbitrum Mainnet   | 4949039107694359620   |
| Arbitrum Sepolia   | 3478487232816046354   |
| Optimism Mainnet   | 7934706634085862462   |
| Optimism Sepolia   | 4547669296748201845   |
| Polygon Mainnet    | 4051577828743386545   |
| Polygon Mumbai     | 12532609583862916517  |
| Avalanche Mainnet  | 6433500567565415111   |
| Avalanche Fuji     | 14767432559492292930  |
| Base Mainnet       | 15971525489660198786  |
| Base Sepolia       | 5795021415249229034   |

ROUTER ADDRESSES:
- Sepolia: 0xD0daae2231A9CB0c823D8aba1E7457f92b587781
- Mainnet: 0x80226fc0Ee2b096224EeAc085Bb9a8cba107A3b9
""")


CHAIN SELECTORS CHAINLINK CCIP

| Chain              | Selector              |
|--------------------|-----------------------|
| Ethereum Mainnet   | 5009297550715157      |
| Ethereum Sepolia   | 16015286601757825753  |
| Arbitrum Mainnet   | 4949039107694359620   |
| Arbitrum Sepolia   | 3478487232816046354   |
| Optimism Mainnet   | 7934706634085862462   |
| Optimism Sepolia   | 4547669296748201845   |
| Polygon Mainnet    | 4051577828743386545   |
| Polygon Mumbai     | 12532609583862916517  |
| Avalanche Mainnet  | 6433500567565415111   |
| Avalanche Fuji     | 14767432559492292930  |
| Base Mainnet       | 15971525489660198786  |
| Base Sepolia       | 5795021415249229034   |

ROUTER ADDRESSES:
- Sepolia: 0xD0daae2231A9CB0c823D8aba1E7457f92b587781
- Mainnet: 0x80226fc0Ee2b096224EeAc085Bb9a8cba107A3b9



---

## 4. Securite Cross-Chain

In [5]:
# Securite cross-chain
print("""
VULNERABILITES CROSS-CHAIN

1. REENTRANCY CROSS-CHAIN
   - Une attaque sur une chain peut affecter l'autre
   - Solution: Verifier la finalite avant execution

2. DOUBLE-SPEND
   - Meme tokens utilises sur plusieurs chains
   - Solution: Lock/Burn atomique

3. ORACLE MANIPULATION
   - Fausse information sur l'etat d'une chain
   - Solution: Multiples oracles, verification

4. BRIDGE EXPLOITS
   - Bugs dans les contrats de bridge
   - Solution: Audits, bug bounties, limits

5. GOVERNANCE ATTACKS
   - Manipulation des votes cross-chain
   - Solution: Snapshot voting, delay

BEST PRACTICES:
- Toujours verifier la finalite (confirmations)
- Rate limits sur les transferts
- Emergency pause mechanism
- Audits reguliers
- Monitoring et alertes
""")


VULNERABILITES CROSS-CHAIN

1. REENTRANCY CROSS-CHAIN
   - Une attaque sur une chain peut affecter l'autre
   - Solution: Verifier la finalite avant execution

2. DOUBLE-SPEND
   - Meme tokens utilises sur plusieurs chains
   - Solution: Lock/Burn atomique

3. ORACLE MANIPULATION
   - Fausse information sur l'etat d'une chain
   - Solution: Multiples oracles, verification

4. BRIDGE EXPLOITS
   - Bugs dans les contrats de bridge
   - Solution: Audits, bug bounties, limits

5. GOVERNANCE ATTACKS
   - Manipulation des votes cross-chain
   - Solution: Snapshot voting, delay

BEST PRACTICES:
- Toujours verifier la finalite (confirmations)
- Rate limits sur les transferts
- Emergency pause mechanism
- Audits reguliers
- Monitoring et alertes



---

## 5. Resume

In [6]:
# Resume
print("""
RESUME CROSS-CHAIN

| Protocol      | Type         | Securite          |
|--------------|--------------|-------------------|
| Chainlink CCIP | Oracle-based | Haute (don-dees) |
| LayerZero     | Ultra-light  | Oracle + Relayer  |
| Axelar        | Gateway      | Proof-of-Stake    |
| Wormhole      | Guardians    | 19/19 signatures  |
| Hyperlane     | Permissionless| Economic security|

QUAND UTILISER:
- CCIP: Pour securite maximale, tokens transfers
- LayerZero: Pour rapidite, omnichain dApps
- Axelar: Pour ecosystem cosmos + EVM
- Wormhole: Pour Solana integration

---

**Notebook suivant** : [SC-15-Final-Project](../05-Capstone/SC-15-Final-Project.ipynb)
""")


RESUME CROSS-CHAIN

| Protocol      | Type         | Securite          |
|--------------|--------------|-------------------|
| Chainlink CCIP | Oracle-based | Haute (don-dees) |
| LayerZero     | Ultra-light  | Oracle + Relayer  |
| Axelar        | Gateway      | Proof-of-Stake    |
| Wormhole      | Guardians    | 19/19 signatures  |
| Hyperlane     | Permissionless| Economic security|

QUAND UTILISER:
- CCIP: Pour securite maximale, tokens transfers
- LayerZero: Pour rapidite, omnichain dApps
- Axelar: Pour ecosystem cosmos + EVM
- Wormhole: Pour Solana integration

---

**Notebook suivant** : [SC-15-Final-Project](../05-Capstone/SC-15-Final-Project.ipynb)



## Exercice : Bridge Token Cross-Chain

Implementez un bridge simple entre deux chaines simulees.

### Objectifs
1. Comprendre le pattern Lock & Mint
2. Implementer un bridge securise
3. Gerer les cas d'erreur et reverts
4. Tester le flux complet

### Instructions

```solidity
// TODO: Implementez un TokenBridge sur la chaine source
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

contract TokenBridge {
    // TODO: Variables d'etat
    // - mapping locked: user => amount
    // - nonce pour unique tx
    // - authorized validator
    
    // TODO: Event Lock(address user, uint256 amount, uint256 nonce)
    // TODO: Event Unlock(address user, uint256 amount, uint256 nonce)
    
    // TODO: lock(amount) - verrouille les tokens de l'utilisateur
    // Emite Lock pour que le bridge detecte
    
    // TODO: unlock(user, amount, nonce, signature) - appele par validator
    // Verifie la signature et debloque
    
    // TODO: validateSignature(...) - verification ECDSA
}
```

```solidity
// TODO: Implementez un WrappedToken sur la chaine destination
// Quand Lock detecte sur source -> Mint sur dest
// Quand Burn sur dest -> Unlock sur source

contract WrappedToken {
    // TODO: mint(user, amount, nonce) - seulement par bridge
    // TODO: burn(user, amount) - utilisateur appelle pour retourner
}
```

### Tests a ecrire
```solidity
// TODO: testLockEmitsEvent
// TODO: testUnlockOnlyValidator
// TODO: testSignatureReplayProtection
// TODO: testFullBridgeRoundTrip
```

### Questions d'analyse
- Comment empecher les attaques par rejeu (replay attacks) ?
- Quelle est la latence minimale pour une transaction cross-chain ?
- Comment gerer les echecs sur la chaine destination ?